In [1]:
import os
import ctypes
import torch

os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]
os.environ["LD_LIBRARY_PATH"] = os.path.expanduser("~/.local/lib") + ":" + os.environ.get("LD_LIBRARY_PATH", "")

LIB = os.path.expanduser("~/.local/lib/libespeak-ng.so")
ctypes.CDLL(LIB)

os.environ["CUDA_VISIBLE_DEVICES"] = "7"
device = torch.device("cuda")

In [2]:
!espeak-ng --version

eSpeak NG text-to-speech: 1.52.0  Data at: /home/minh_duc/.local/share/espeak-ng-data


In [9]:
from neutts import NeuTTS
import soundfile as sf

tts = NeuTTS(
   backbone_repo="neuphonic/neutts-nano", # or 'neutts-nano-q4-gguf' with llama-cpp-python installed
   backbone_device="cuda",
   codec_repo="neuphonic/neucodec",
   codec_device="cuda"
)
input_text = "Twas brillig, and the slithy toves Did gyre and gimble in the wabe; All mimsy were the borogoves, And the mome raths outgrabe."

ref_text = "samples/jo.txt"
ref_audio_path = "samples/jo.wav"

ref_text = open(ref_text, "r").read().strip()
ref_codes = tts.encode_reference(ref_audio_path)

wav, codes = tts.infer(input_text, ref_codes, ref_text, return_codes=True)
sf.write("test.wav", wav, 24000)

Loading phonemizer...
Loading backbone from: neuphonic/neutts-nano on cuda ...
Loading codec from: neuphonic/neucodec on cuda ...


Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00, 14873.42it/s]
/home/minh_duc/miniconda3/envs/neutts_env/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


loaded PerthNet (Implicit) at step 250,000


In [10]:
codes

'<|speech_50534|><|speech_38978|><|speech_21862|><|speech_21590|><|speech_21846|><|speech_6230|><|speech_5479|><|speech_7271|><|speech_5542|><|speech_6246|><|speech_52354|><|speech_28985|><|speech_8203|><|speech_43337|><|speech_60509|><|speech_56472|><|speech_65270|><|speech_48048|><|speech_30652|><|speech_29277|><|speech_43996|><|speech_45918|><|speech_49100|><|speech_29535|><|speech_48076|><|speech_45914|><|speech_30604|><|speech_29534|><|speech_41932|><|speech_38745|><|speech_25412|><|speech_16210|><|speech_48657|><|speech_64021|><|speech_60969|><|speech_60206|><|speech_60309|><|speech_53158|><|speech_52456|><|speech_52418|><|speech_59798|><|speech_65261|><|speech_61312|><|speech_56206|><|speech_33557|><|speech_38665|><|speech_39733|><|speech_60349|><|speech_44921|><|speech_64285|><|speech_63273|><|speech_62233|><|speech_45846|><|speech_45677|><|speech_45849|><|speech_29549|><|speech_29225|><|speech_26559|><|speech_58994|><|speech_57494|><|speech_62138|><|speech_50872|><|speech_5041

In [11]:
import re
print(re.sub(r"<\|speech_\d+\|>", "", codes))


<|SPEECH_GENERATION_END|>


In [3]:
wav.shape

(330720,)

In [4]:
code = tts.encode_reference(ref_audio_path="./test.wav")

In [11]:
print(code)
code.shape

tensor([22742,  4498, 23653, 20906,  2214, 50614, 55027, 33681, 33625,  5889,
        44067, 56854, 26136, 59592, 59117, 27116, 13311, 29612, 43800, 42925,
        42927, 55278, 35644, 42908, 47912, 16204, 44345, 44408, 40484, 40308,
        20024, 51566, 58255, 63337, 15214, 44986, 52405, 64710, 31556, 46041,
        13825, 13286, 39463, 58999, 61939, 31318, 32218, 22179, 24539,   533,
        17306, 51047, 22491, 59814, 56800, 62789, 61902, 37383, 53081, 50122,
        18252,   394, 51036, 21240, 34493, 43972, 35500, 40452, 43484,  3152,
        28137, 56726, 52390, 57066, 18752, 16844, 57608, 38620,   260, 42197,
        50373, 49322, 33384, 42750, 37234, 56807, 34530, 50642, 17818, 20681,
        46235, 30089, 16731,   504,   773, 31378, 55314, 51554, 45410, 30965,
        37091,  3818, 34787, 64337, 33377, 29329,   850, 43539, 60466, 56609,
        53505, 49490, 51954, 22706, 54897, 54586, 35130, 51515, 60023, 15219,
         1647, 19259, 18239, 55867, 50551, 36342, 64725, 48168, 

torch.Size([689])

In [6]:
print(max(code))

tensor(65502, device='cuda:0', dtype=torch.int32)


In [7]:
def speech_ids_to_codes(speech_ids):
    # Handle torch tensor
    if hasattr(speech_ids, "tolist"):
        speech_ids = speech_ids.tolist()

    return "".join(f"<|speech_{i}|>" for i in speech_ids)


In [8]:
code_str = speech_ids_to_codes(code)

In [9]:
len(code_str)

10782

In [10]:
code.shape

torch.Size([689])

In [12]:
de_wav = tts._decode(code_str)

In [11]:
code_str

'<|speech_50330|><|speech_33814|><|speech_22681|><|speech_21866|><|speech_6370|><|speech_53670|><|speech_51638|><|speech_18034|><|speech_1898|><|speech_17156|><|speech_24083|><|speech_52503|><|speech_18293|><|speech_19100|><|speech_13887|><|speech_12781|><|speech_31341|><|speech_47021|><|speech_47019|><|speech_64510|><|speech_63277|><|speech_29372|><|speech_13869|><|speech_15945|><|speech_43294|><|speech_43305|><|speech_28004|><|speech_36197|><|speech_23653|><|speech_54570|><|speech_45707|><|speech_13666|><|speech_15135|><|speech_27509|><|speech_57003|><|speech_51349|><|speech_65501|><|speech_46916|><|speech_14302|><|speech_9810|><|speech_42551|><|speech_54589|><|speech_54394|><|speech_58995|><|speech_14071|><|speech_30994|><|speech_20447|><|speech_7139|><|speech_18398|><|speech_50005|><|speech_17308|><|speech_675|><|speech_9946|><|speech_59878|><|speech_52466|><|speech_62610|><|speech_61658|><|speech_37455|><|speech_52697|><|speech_49802|><|speech_1676|><|speech_715|><|speech_7148|><|

In [13]:
from IPython.display import Audio, display

display(Audio(data=de_wav, rate=24000))


In [14]:
tts.backbone

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(194256, 576, padding_idx=128001)
    (layers): ModuleList(
      (0-23): 24 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=576, out_features=576, bias=False)
          (k_proj): Linear(in_features=576, out_features=192, bias=False)
          (v_proj): Linear(in_features=576, out_features=192, bias=False)
          (o_proj): Linear(in_features=576, out_features=576, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=576, out_features=2304, bias=False)
          (up_proj): Linear(in_features=576, out_features=2304, bias=False)
          (down_proj): Linear(in_features=2304, out_features=576, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((576,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((576,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((576,), eps=1e-05)
    (rotar

In [16]:
tts.codec

NeuCodec(
  (semantic_model): Wav2Vec2BertModel(
    (feature_projection): Wav2Vec2BertFeatureProjection(
      (layer_norm): LayerNorm((160,), eps=1e-05, elementwise_affine=True)
      (projection): Linear(in_features=160, out_features=1024, bias=True)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): Wav2Vec2BertEncoder(
      (dropout): Dropout(p=0.0, inplace=False)
      (layers): ModuleList(
        (0-23): 24 x Wav2Vec2BertEncoderLayer(
          (ffn1_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (ffn1): Wav2Vec2BertFeedForward(
            (intermediate_dropout): Dropout(p=0.0, inplace=False)
            (intermediate_dense): Linear(in_features=1024, out_features=4096, bias=True)
            (intermediate_act_fn): SiLU()
            (output_dense): Linear(in_features=4096, out_features=1024, bias=True)
            (output_dropout): Dropout(p=0.0, inplace=False)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps